In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 쇼어 알고리즘

*사용 예상 시간: Eagle r3 프로세서에서 약 3초 (참고: 이는 예상치이며, 실제 실행 시간은 다를 수 있습니다.)*

[쇼어 알고리즘](https://epubs.siam.org/doi/abs/10.1137/S0036144598347011)은 1994년 Peter Shor가 개발한 것으로, 정수를 다항식 시간 내에 인수분해하는 획기적인 양자 알고리즘입니다. 이 알고리즘의 중요성은 알려진 어떠한 고전 알고리즘보다 지수적으로 빠르게 큰 정수를 인수분해할 수 있다는 점에 있으며, 이는 큰 수의 인수분해가 어렵다는 점에 의존하는 RSA와 같은 광범위하게 사용되는 암호화 시스템의 보안을 위협합니다. 충분히 강력한 양자 컴퓨터에서 이 문제를 효율적으로 해결함으로써 쇼어 알고리즘은 암호학, 사이버 보안, 계산 수학 등의 분야를 혁신할 수 있으며, 이는 양자 계산의 변혁적인 힘을 잘 보여줍니다.

이 튜토리얼은 양자 컴퓨터에서 15를 인수분해함으로써 쇼어 알고리즘을 시연하는 데 초점을 맞춥니다.

먼저 위수 찾기 문제(order finding problem)를 정의하고 양자 위상 추정 프로토콜로부터 해당 Circuit을 구성합니다. 다음으로, 트랜스파일할 수 있는 가장 짧은 깊이의 Circuit을 사용하여 실제 하드웨어에서 위수 찾기 Circuit을 실행합니다. 마지막 섹션에서는 위수 찾기 문제를 정수 인수분해와 연결하여 쇼어 알고리즘을 완성합니다.

튜토리얼 마지막 부분에서는 실제 하드웨어에서의 다른 쇼어 알고리즘 시연 사례들을 살펴보며, 범용 구현과 15나 21과 같은 특정 정수의 인수분해에 맞춰진 구현 모두에 초점을 맞춥니다.
참고: 이 튜토리얼은 쇼어 알고리즘과 관련된 Circuit의 구현 및 시연에 더 집중합니다. 해당 내용에 대한 심층적인 교육 자료는 John Watrous 박사의 [양자 알고리즘 기초](/learning/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/introduction) 강좌와 [참고문헌](#references) 섹션의 논문들을 참조하시기 바랍니다.
### 요구 사항
이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하십시오:
- [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원이 포함된 Qiskit SDK v2.0 이상
- Qiskit Runtime v0.40 이상 (`pip install qiskit-ibm-runtime`)
### 설정

In [ ]:
import numpy as np
import pandas as pd
from fractions import Fraction
from math import floor, gcd, log

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager
from qiskit.visualization import plot_histogram

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## 1단계: 고전적 입력을 양자 문제로 변환하기
### 배경
쇼어의 정수 인수분해 알고리즘은 *위수 찾기* 문제라는 중간 문제를 활용합니다. 이 섹션에서는 *양자 위상 추정*을 사용하여 위수 찾기 문제를 푸는 방법을 시연합니다.
### 위상 추정 문제
위상 추정 문제에서는 $n$개의 Qubit으로 이루어진 양자 상태 $\ket{\psi}$와 $n$개의 Qubit에 작용하는 유니터리 양자 Circuit이 주어집니다. $\ket{\psi}$는 해당 Circuit의 작용을 설명하는 유니터리 행렬 $U$의 고유벡터임이 보장되며, 우리의 목표는 $\ket{\psi}$에 대응하는 고유값 $\lambda = e^{2 \pi i \theta}$를 계산하거나 근사하는 것입니다. 다시 말해, Circuit은 다음을 만족하는 $\theta \in [0, 1)$에 대한 근삿값을 출력해야 합니다. $$U \ket{\psi}= e^{2 \pi i \theta} \ket{\psi}.$$
위상 추정 Circuit의 목표는 $m$비트로 $\theta$를 근사하는 것입니다. 수학적으로는 $\theta \approx y / 2^m$을 만족하는 $y$를 찾고자 하며, 여기서 $y \in {0, 1, 2, \dots, 2^{m-1}}$입니다. 다음 이미지는 $m$개의 Qubit에 대한 측정을 수행하여 $m$비트로 $y$를 추정하는 양자 Circuit을 보여줍니다.
![양자 위상 추정 Circuit](../learning/images/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/phase-estimation-procedure.svg)
위 Circuit에서 상위 $m$개의 Qubit은 $\ket{0^m}$ 상태로 초기화되고, 하위 $n$개의 Qubit은 $U$의 고유벡터임이 보장된 $\ket{\psi}$로 초기화됩니다. 위상 추정 Circuit의 첫 번째 구성 요소는 제어 Qubit으로 *위상 킥백(phase kickback)*을 수행하는 제어-유니터리 연산입니다. 이 제어 유니터리들은 최하위 비트부터 최상위 비트까지 제어 Qubit의 위치에 따라 지수적으로 적용됩니다. $\ket{\psi}$가 $U$의 고유벡터이므로, 하위 $n$개의 Qubit 상태는 이 연산에 의해 영향을 받지 않지만, 고유값의 위상 정보는 상위 $m$개의 Qubit으로 전파됩니다.
제어 유니터리를 통한 위상 킥백 연산 이후, 상위 $m$개의 Qubit의 모든 가능한 상태는 유니터리 $U$의 각 고유벡터 $\ket{\psi}$에 대해 서로 정규직교함이 밝혀져 있습니다. 따라서 이 상태들은 완벽하게 구별 가능하며, 측정을 위해 이들이 형성하는 기저를 계산 기저로 다시 회전시킬 수 있습니다. 수학적 분석에 따르면 이 회전 행렬은 $2^m$차원 힐베르트 공간에서의 역 양자 푸리에 변환(QFT)에 해당합니다. 이에 대한 직관은, 모듈러 지수 연산자의 주기적 구조가 양자 상태에 인코딩되고, QFT가 이 주기성을 주파수 도메인의 측정 가능한 피크로 변환한다는 것입니다.

쇼어 알고리즘에서 QFT Circuit이 사용되는 이유에 대한 더 깊은 이해를 위해서는 [양자 알고리즘 기초](/learning/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/introduction) 강좌를 참조하시기 바랍니다.
이제 위수 찾기에 위상 추정 Circuit을 사용할 준비가 되었습니다.
### 위수 찾기 문제
위수 찾기 문제를 정의하기 위해 몇 가지 수론 개념부터 시작합니다. 먼저, 임의의 양의 정수 $N$에 대해 집합 $\mathbb{Z}_N$을 다음과 같이 정의합니다. $$\mathbb{Z}_N = {0, 1, 2, \dots, N-1}.$$
$\mathbb{Z}_N$에서의 모든 산술 연산은 $N$을 법으로(modulo $N$) 수행됩니다. 특히, $N$과 서로소인 모든 원소 $a \in \mathbb{Z}_n$은 특별하며 다음과 같이 $\mathbb{Z}^*_N$을 구성합니다. $$\mathbb{Z}^*_N = { a \in \mathbb{Z}_N : \mathrm{gcd}(a, N)=1 }.$$
원소 $a \in \mathbb{Z}^*_N$에 대해 $$a^r \equiv 1 \; (\mathrm{mod} \; N)$$을 만족하는 가장 작은 양의 정수 $r$을 $a$의 $N$에 대한 *위수(order)*라고 정의합니다. 나중에 살펴보겠지만, $a \in \mathbb{Z}^*_N$의 위수를 찾으면 $N$을 인수분해할 수 있습니다.
위상 추정 Circuit으로부터 위수 찾기 Circuit을 구성하기 위해서는 두 가지를 고려해야 합니다. 첫째, 위수 $r$을 찾을 수 있게 해주는 유니터리 $U$를 정의해야 하고, 둘째, 위상 추정 Circuit의 초기 상태를 준비하기 위한 $U$의 고유벡터 $\ket{\psi}$를 정의해야 합니다.

위수 찾기 문제를 위상 추정과 연결하기 위해, 고전적 상태가 $\mathbb{Z}_N$에 대응하는 시스템에 정의된 연산을 고려합니다. 여기서 고정된 원소 $a \in \mathbb{Z}^*_N$을 곱합니다. 특히, 각 $x \in \mathbb{Z}_N$에 대해 $$M_a \ket{x} = \ket{ax \; (\mathrm{mod} \; N)}$$이 되도록 이 곱셈 연산자 $M_a$를 정의합니다. 방정식 우변의 ket 내부에서 $N$을 법으로 곱을 취한다는 것이 암묵적으로 포함되어 있음에 주의하십시오. 수학적 분석에 따르면 $M_a$는 유니터리 연산자입니다. 더 나아가, $M_a$는 $a$의 위수 $r$을 위상 추정 문제와 연결할 수 있게 해주는 고유벡터 및 고유값 쌍을 가집니다. 구체적으로, 임의의 $j \in {0, \dots, r-1}$ 선택에 대해 $$\ket{\psi_j} = \frac{1}{\sqrt{r}} \sum^{r-1}_{k=0} \omega^{-jk}_{r} \ket{a^k}$$는 대응하는 고유값이 $\omega^{j}_{r}$인 $M_a$의 고유벡터입니다. 여기서 $$\omega^{j}_{r} = e^{2 \pi i \frac{j}{r}}.$$
관찰에 의하면, 편리한 고유벡터/고유값 쌍은 $\omega^{1}_{r} = e^{2 \pi i \frac{1}{r}}$을 갖는 상태 $\ket{\psi_1}$임을 알 수 있습니다. 따라서 고유벡터 $\ket{\psi_1}$를 찾을 수 있다면, 양자 Circuit으로 위상 $\theta=1/r$을 추정하여 위수 $r$의 추정치를 얻을 수 있습니다. 하지만 이것이 쉽지 않으므로 대안을 고려해야 합니다.

계산 기저 상태 $\ket{1}$을 초기 상태로 준비하면 Circuit이 어떤 결과를 낼지 생각해 보겠습니다. 이것은 $M_a$의 고유 상태가 아니지만, 앞서 설명한 고유 상태들의 균일한 중첩입니다. 다시 말해, 다음 관계가 성립합니다. $$ \ket{1} = \frac{1}{\sqrt{r}} \sum^{r-1}_{k=0} \ket{\psi_k} $$
위 방정식의 의미는, 초기 상태를 $\ket{1}$로 설정하면 $k \in { 0, \dots, r-1}$을 균일하게 임의로 선택하여 위상 추정 Circuit에서 고유벡터로 $\ket{\psi_k}$를 사용한 것과 정확히 동일한 측정 결과를 얻는다는 것입니다. 즉, 상위 $m$개의 Qubit에 대한 측정은 균일하게 임의로 선택된 $k \in { 0, \dots, r-1}$에서 값 $k / r$에 대한 근삿값 $y / 2^m$을 산출합니다. 이를 통해 여러 번의 독립적인 실행 후 높은 신뢰도로 $r$을 알아낼 수 있으며, 이것이 우리의 목표였습니다.
### 모듈러 지수 연산자
지금까지 우리는 양자 Circuit에서 $U = M_a$와 $\ket{\psi} = \ket{1}$을 정의함으로써 위상 추정 문제를 위수 찾기 문제와 연결했습니다. 따라서 마지막으로 남은 것은 $k = 1, 2, 4, \dots, 2^{m-1}$에 대해 $M_a^k$로서 $M_a$의 모듈러 지수를 효율적으로 정의하는 방법을 찾는 것입니다.
이 계산을 수행하기 위해, 선택한 임의의 거듭제곱 $k$에 대해 $M_a$ Circuit을 $k$번 반복하는 대신 $b = a^k \; \mathrm{mod} \; N$을 계산한 다음 $M_b$ Circuit을 사용하여 $M_a^k$에 대한 Circuit을 생성할 수 있음을 알 수 있습니다. 2의 거듭제곱에 해당하는 멱수만 필요하므로, 반복 제곱법을 사용하여 이를 고전적으로 효율적으로 수행할 수 있습니다.
## 2단계: 양자 하드웨어 실행을 위한 문제 최적화
### $N = 15$, $a=2$를 이용한 구체적 예시
여기서 잠시 구체적인 예시를 살펴보고 $N=15$에 대한 위수 찾기 Circuit을 구성해 보겠습니다. $N=15$에 대해 가능한 비자명 원소 $a \in \mathbb{Z}_N^*$는 $a \in {2, 4, 7, 8, 11, 13, 14 }$임에 유의하십시오. 이 예시에서는 $a=2$를 선택합니다. $M_2$ 연산자와 모듈러 지수 연산자 $M_2^k$를 구성하겠습니다.
$M_2$의 계산 기저 상태에 대한 작용은 다음과 같습니다.
$$M_2 \ket{0} = \ket{0} \quad M_2 \ket{5} = \ket{10} \quad M_2 \ket{10} = \ket{5}$$
$$M_2 \ket{1} = \ket{2} \quad M_2 \ket{6} = \ket{12} \quad M_2 \ket{11} = \ket{7}$$
$$M_2 \ket{2} = \ket{4} \quad M_2 \ket{7} = \ket{14} \quad M_2 \ket{12} = \ket{9}$$
$$M_2 \ket{3} = \ket{6} \quad M_2 \ket{8} = \ket{1} \quad M_2 \ket{13} = \ket{11}$$
$$M_2 \ket{4} = \ket{8} \quad M_2 \ket{9} = \ket{3} \quad M_2 \ket{14} = \ket{13}$$
관찰에 의하면 기저 상태들이 뒤섞이므로 순열 행렬임을 알 수 있습니다. 이 연산을 스왑 Gate를 사용하여 4개의 Qubit으로 구성할 수 있습니다. 아래에서 $M_2$와 제어-$M_2$ 연산을 구성합니다.

In [2]:
def M2mod15():
    """
    M2 (mod 15)
    """
    b = 2
    U = QuantumCircuit(4)

    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)

    U = U.to_gate()
    U.name = f"M_{b}"

    return U

In [3]:
# Get the M2 operator
M2 = M2mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M2, inplace=True)
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0a8885f1-91d4-40bd-912d-dc5eea05f5bd-0.avif" alt="Output of the previous code cell" />

In [4]:
def controlled_M2mod15():
    """
    Controlled M2 (mod 15)
    """
    b = 2
    U = QuantumCircuit(4)

    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)

    U = U.to_gate()
    U.name = f"M_{b}"
    c_U = U.control()

    return c_U

In [5]:
# Get the controlled-M2 operator
controlled_M2 = controlled_M2mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(5)
circ.compose(controlled_M2, inplace=True)
circ.decompose(reps=1).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/ab7fe331-2f9e-47ca-ba3b-f5d67992062a-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/ab7fe331-2f9e-47ca-ba3b-f5d67992062a-0.avif)

2개 이상의 Qubit에 작용하는 Gate는 2-Qubit Gate로 추가 분해됩니다.

In [6]:
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/13b4841d-a4ac-46bd-b4d0-d111b3017189-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/13b4841d-a4ac-46bd-b4d0-d111b3017189-0.avif)

이제 모듈러 지수 연산자를 구성해야 합니다. 위상 추정에서 충분한 정밀도를 얻기 위해 추정 측정에 8개의 Qubit을 사용합니다. 따라서 각 $k = 0, 1, \dots, 7$에 대해 $b = a^{2^k} \; (\mathrm{mod} \; N)$을 갖는 $M_b$를 구성해야 합니다.

In [7]:
def a2kmodN(a, k, N):
    """Compute a^{2^k} (mod N) by repeated squaring"""
    for _ in range(k):
        a = int(np.mod(a**2, N))
    return a

In [8]:
k_list = range(8)
b_list = [a2kmodN(2, k, 15) for k in k_list]

print(b_list)

[2, 4, 1, 1, 1, 1, 1, 1]


As we can see from the list of $b$ values, in addition to $M_2$ that we previously constructed, we also need to build $M_4$ and $M_1$. Note that $M_1$ acts trivially on the computational basis states, so it is simply the identity operator.

$M_4$ acts on the computational basis states as follows.
$$M_4 \ket{0} = \ket{0} \quad M_4 \ket{5} = \ket{5} \quad M_4 \ket{10} = \ket{10}$$
$$M_4 \ket{1} = \ket{4} \quad M_4 \ket{6} = \ket{9} \quad M_4 \ket{11} = \ket{14}$$
$$M_4 \ket{2} = \ket{8} \quad M_4 \ket{7} = \ket{13} \quad M_4 \ket{12} = \ket{3}$$
$$M_4 \ket{3} = \ket{12} \quad M_4 \ket{8} = \ket{2} \quad M_4 \ket{13} = \ket{7}$$
$$M_4 \ket{4} = \ket{1} \quad M_4 \ket{9} = \ket{6} \quad M_4 \ket{14} = \ket{11}$$

Therefore, this permutation can be constructed with the following swap operation.

In [9]:
def M4mod15():
    """
    M4 (mod 15)
    """
    b = 4
    U = QuantumCircuit(4)

    U.swap(1, 3)
    U.swap(0, 2)

    U = U.to_gate()
    U.name = f"M_{b}"

    return U

In [10]:
# Get the M4 operator
M4 = M4mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M4, inplace=True)
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/be041e3d-28b1-453e-983e-184c2366aeb9-0.avif" alt="Output of the previous code cell" />

In [11]:
def controlled_M4mod15():
    """
    Controlled M4 (mod 15)
    """
    b = 4
    U = QuantumCircuit(4)

    U.swap(1, 3)
    U.swap(0, 2)

    U = U.to_gate()
    U.name = f"M_{b}"
    c_U = U.control()

    return c_U

In [12]:
# Get the controlled-M4 operator
controlled_M4 = controlled_M4mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(5)
circ.compose(controlled_M4, inplace=True)
circ.decompose(reps=1).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/8d943b00-a502-4157-8a0d-13fb1f55e705-0.avif" alt="Output of the previous code cell" />

Gates acting on more than two qubits will be further decomposed into two-qubit gates.

In [13]:
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/68399eef-5e55-4c95-a8a4-c8efaebd34b9-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/8d943b00-a502-4157-8a0d-13fb1f55e705-0.avif)

두 큐비트 이상에 작용하는 Gate는 추가로 2-큐비트 Gate로 분해됩니다.

In [14]:
def mod_mult_gate(b, N):
    """
    Modular multiplication gate from permutation matrix.
    """
    if gcd(b, N) > 1:
        print(f"Error: gcd({b},{N}) > 1")
    else:
        n = floor(log(N - 1, 2)) + 1
        U = np.full((2**n, 2**n), 0)
        for x in range(N):
            U[b * x % N][x] = 1
        for x in range(N, 2**n):
            U[x][x] = 1
        G = UnitaryGate(U)
        G.name = f"M_{b}"
        return G

In [15]:
# Let's build M2 using the permutation matrix definition
M2_other = mod_mult_gate(2, 15)

# Add it to a circuit
circ = QuantumCircuit(4)
circ.compose(M2_other, inplace=True)
circ = circ.decompose()

# Transpile the circuit and get the depth
coupling_map = CouplingMap.from_line(4)
pm = generate_preset_pass_manager(coupling_map=coupling_map)
transpiled_circ = pm.run(circ)

print(f"qubits: {circ.num_qubits}")
print(
    f"2q-depth: {transpiled_circ.depth(lambda x: x.operation.num_qubits==2)}"
)
print(f"2q-size: {transpiled_circ.size(lambda x: x.operation.num_qubits==2)}")
print(f"Operator counts: {transpiled_circ.count_ops()}")
transpiled_circ.decompose().draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

qubits: 4
2q-depth: 94
2q-size: 96
Operator counts: OrderedDict({'cx': 45, 'swap': 32, 'u': 24, 'u1': 7, 'u3': 4, 'unitary': 3, 'circuit-335': 1, 'circuit-338': 1, 'circuit-341': 1, 'circuit-344': 1, 'circuit-347': 1, 'circuit-350': 1, 'circuit-353': 1, 'circuit-356': 1, 'circuit-359': 1, 'circuit-362': 1, 'circuit-365': 1, 'circuit-368': 1, 'circuit-371': 1, 'circuit-374': 1, 'circuit-377': 1, 'circuit-380': 1})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/c184f6dd-9f80-4487-ac0b-0dd94170b0f0-1.avif" alt="Output of the previous code cell" />

Let's compare these counts with the compiled circuit depth of our manual implementation of the $M_2$ gate.

In [16]:
# Get the M2 operator from our manual construction
M2 = M2mod15()

# Add it to a circuit
circ = QuantumCircuit(4)
circ.compose(M2, inplace=True)
circ = circ.decompose(reps=3)

# Transpile the circuit and get the depth
coupling_map = CouplingMap.from_line(4)
pm = generate_preset_pass_manager(coupling_map=coupling_map)
transpiled_circ = pm.run(circ)

print(f"qubits: {circ.num_qubits}")
print(
    f"2q-depth: {transpiled_circ.depth(lambda x: x.operation.num_qubits==2)}"
)
print(f"2q-size: {transpiled_circ.size(lambda x: x.operation.num_qubits==2)}")
print(f"Operator counts: {transpiled_circ.count_ops()}")
transpiled_circ.draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

qubits: 4
2q-depth: 9
2q-size: 9
Operator counts: OrderedDict({'cx': 9})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0235c931-0adb-4972-9fce-32a0341822bf-1.avif" alt="Output of the previous code cell" />

As we can see, the permutation matrix approach resulted in a significantly deep circuit even for a single $M_2$ gate compared to our manual implementation of it. Therefore, we will continue with our previous implementation of the $M_b$ operations.

Now, we are ready to construct the full order finding circuit using our previously defined controlled modular exponentiation operators. In the following code, we also import the [QFT circuit](/docs/api/qiskit/qiskit.circuit.library.QFT) from the Qiskit Circuit library, which uses Hadamard gates on each qubit, a series of controlled-U1 (or Z, depending on the phase) gates, and a layer of swap gates.

In [17]:
# Order finding problem for N = 15 with a = 2
N = 15
a = 2

# Number of qubits
num_target = floor(log(N - 1, 2)) + 1  # for modular exponentiation operators
num_control = 2 * num_target  # for enough precision of estimation

# List of M_b operators in order
k_list = range(num_control)
b_list = [a2kmodN(2, k, 15) for k in k_list]

# Initialize the circuit
control = QuantumRegister(num_control, name="C")
target = QuantumRegister(num_target, name="T")
output = ClassicalRegister(num_control, name="out")
circuit = QuantumCircuit(control, target, output)

# Initialize the target register to the state |1>
circuit.x(num_control)

# Add the Hadamard gates and controlled versions of the
# multiplication gates
for k, qubit in enumerate(control):
    circuit.h(k)
    b = b_list[k]
    if b == 2:
        circuit.compose(
            M2mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    elif b == 4:
        circuit.compose(
            M4mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    else:
        continue  # M1 is the identity operator

# Apply the inverse QFT to the control register
circuit.compose(QFT(num_control, inverse=True), qubits=control, inplace=True)

# Measure the control register
circuit.measure(control, output)

circuit.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0e854aed-c11b-494c-8c80-adeb8eb0e8fe-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/c184f6dd-9f80-4487-ac0b-0dd94170b0f0-1.avif)

이 카운트를 $M_2$ Gate의 수동 구현 방식으로 컴파일된 Circuit 깊이와 비교해 보겠습니다.

In [ ]:
service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh")
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)

transpiled_circuit = pm.run(circuit)

print(
    f"2q-depth: {transpiled_circuit.depth(lambda x: x.operation.num_qubits==2)}"
)
print(
    f"2q-size: {transpiled_circuit.size(lambda x: x.operation.num_qubits==2)}"
)
print(f"Operator counts: {transpiled_circuit.count_ops()}")
transpiled_circuit.draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

2q-depth: 187
2q-size: 260
Operator counts: OrderedDict({'sx': 521, 'rz': 354, 'cz': 260, 'measure': 8, 'x': 4})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/95925dd5-7ba9-4746-b96e-ba50400fa5ac-1.avif" alt="Output of the previous code cell" />

## Step 3: Execute using Qiskit primitives

First, we discuss what we would theoretically obtain if we ran this circuit on an ideal simulator. Below, we have a set of simulation results of the above circuit using 1024 shots. As we can see, we get an approximately uniform distribution over four bitstrings over the control qubits.

In [19]:
# Obtained from the simulator
counts = {"00000000": 264, "01000000": 268, "10000000": 249, "11000000": 243}

In [20]:
plot_histogram(counts)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0d6d2702-02e4-47de-8f7e-0b256657ef0f-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/0e854aed-c11b-494c-8c80-adeb8eb0e8fe-0.avif)

나머지 제어 Qubit에서의 제어형 모듈러 지수 연산은 $M_1$이 항등(identity) 연산자이므로 생략하였습니다.
이 튜토리얼의 후반부에서는 이 Circuit을 `ibm_marrakesh` Backend에서 실행할 것입니다. 이를 위해 해당 특정 Backend에 맞게 Circuit을 트랜스파일하고, Circuit 깊이 및 Gate 카운트를 보고합니다.

In [21]:
# Rows to be displayed in table
rows = []
# Corresponding phase of each bitstring
measured_phases = []

for output in counts:
    decimal = int(output, 2)  # Convert bitstring to decimal
    phase = decimal / (2**num_control)  # Find corresponding eigenvalue
    measured_phases.append(phase)
    # Add these values to the rows in our table:
    rows.append(
        [
            f"{output}(bin) = {decimal:>3}(dec)",
            f"{decimal}/{2 ** num_control} = {phase:.2f}",
        ]
    )

# Print the rows in a table
headers = ["Register Output", "Phase"]
df = pd.DataFrame(rows, columns=headers)
print(df)

            Register Output           Phase
0  00000000(bin) =   0(dec)    0/256 = 0.00
1  01000000(bin) =  64(dec)   64/256 = 0.25
2  10000000(bin) = 128(dec)  128/256 = 0.50
3  11000000(bin) = 192(dec)  192/256 = 0.75


Recall that the any measured phase corresponds to $\theta = k / r$ where $k$ is sampled uniformly at random from $\{0, 1, \dots, r-1 \}$. Therefore, we can use the continued fractions algorithm to attempt to find $k$ and the order $r$. Python has this functionality built in. We can use the `fractions` module to turn a float into a `Fraction` object, for example:

In [22]:
Fraction(0.666)

Fraction(5998794703657501, 9007199254740992)

![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/95925dd5-7ba9-4746-b96e-ba50400fa5ac-1.avif)
## 3단계: Qiskit 프리미티브를 사용한 실행
먼저, 이 Circuit을 이상적인 시뮬레이터에서 실행했을 때 이론적으로 얻을 수 있는 결과에 대해 논의합니다. 아래에는 1024 shots으로 위 Circuit을 시뮬레이션한 결과가 있습니다. 보시다시피, 제어 Qubit에 대한 네 개의 비트열에 걸쳐 거의 균일한 분포를 얻게 됩니다.

In [23]:
# Get fraction that most closely resembles 0.666
# with denominator < 15
Fraction(0.666).limit_denominator(15)

Fraction(2, 3)

This is much nicer. The order (r) must be less than N, so we will set the maximum denominator to be `15`:

In [24]:
# Rows to be displayed in a table
rows = []

for phase in measured_phases:
    frac = Fraction(phase).limit_denominator(15)
    rows.append(
        [phase, f"{frac.numerator}/{frac.denominator}", frac.denominator]
    )

# Print the rows in a table
headers = ["Phase", "Fraction", "Guess for r"]
df = pd.DataFrame(rows, columns=headers)
print(df)

   Phase Fraction  Guess for r
0   0.00      0/1            1
1   0.25      1/4            4
2   0.50      1/2            2
3   0.75      3/4            4


![이전 코드 셀의 출력](../docs/images/tutorials/shors-algorithm/extracted-outputs/0d6d2702-02e4-47de-8f7e-0b256657ef0f-0.avif)

제어 Qubit을 측정함으로써 $M_a$ 연산자의 8비트 위상 추정값을 얻습니다. 이 이진 표현을 십진수로 변환하여 측정된 위상을 구할 수 있습니다. 위의 히스토그램에서 볼 수 있듯이 네 개의 서로 다른 비트열이 측정되었으며, 각각은 다음과 같이 위상 값에 대응합니다.

In [ ]:
# Sampler primitive to obtain the probability distribution
sampler = Sampler(backend)

# Turn on dynamical decoupling with sequence XpXm
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"
# Enable gate twirling
sampler.options.twirling.enable_gates = True

pub = transpiled_circuit
job = sampler.run([pub], shots=1024)

In [25]:
result = job.result()[0]
counts = result.data["out"].get_counts()

In [26]:
plot_histogram(counts, figsize=(35, 5))

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/559d7030-1f67-44e8-afa7-6afc7a334677-0.avif" alt="Output of the previous code cell" />

As we can see, we obtained the same bitstrings with highest counts. Since quantum hardware has noise, there is some leakage to other bitstrings, which we can filter out statistically.

In [27]:
# Dictionary of bitstrings and their counts to keep
counts_keep = {}
# Threshold to filter
threshold = np.max(list(counts.values())) / 2

for key, value in counts.items():
    if value > threshold:
        counts_keep[key] = value

print(counts_keep)

{'00000000': 58, '01000000': 41, '11000000': 42, '10000000': 40}


이 방법은 결과를 정확하게 반환하는 분수를 제공하기 때문에(이 경우 `0.6660000...`), 위와 같이 복잡한 결과가 나올 수 있습니다. `.limit_denominator()` 메서드를 사용하면 분모가 특정 값 이하이면서 부동소수점 수에 가장 근접한 분수를 구할 수 있습니다:

In [28]:
a = 2
N = 15

FACTOR_FOUND = False
num_attempt = 0

while not FACTOR_FOUND:
    print(f"\nATTEMPT {num_attempt}:")
    # Here, we get the bitstring by iterating over outcomes
    # of a previous hardware run with multiple shots.
    # Instead, we can also perform a single-shot measurement
    # here in the loop.
    bitstring = list(counts_keep.keys())[num_attempt]
    num_attempt += 1
    # Find the phase from measurement
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control)  # phase = k / r
    print(f"Phase: theta = {phase}")

    # Guess the order from phase
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator  # order = r
    print(f"Order of {a} modulo {N} estimated as: r = {r}")

    if phase != 0:
        # Guesses for factors are gcd(a^{r / 2} ± 1, 15)
        if r % 2 == 0:
            x = pow(a, r // 2, N) - 1
            d = gcd(x, N)
            if d > 1:
                FACTOR_FOUND = True
                print(f"*** Non-trivial factor found: {x} ***")


ATTEMPT 0:
Phase: theta = 0.0
Order of 2 modulo 15 estimated as: r = 1

ATTEMPT 1:
Phase: theta = 0.25
Order of 2 modulo 15 estimated as: r = 4
*** Non-trivial factor found: 3 ***


## Discussion

### Related work
In this section, we discuss other milestone work that has demonstrated Shor's algorithm on real hardware.

The seminal work [[3]](#references) from IBM&reg; demonstrated Shor's algorithm for the first time, factoring the number 15 into its prime factors 3 and 5 using a seven-qubit nuclear magnetic resonance (NMR) quantum computer. Another experiment [[4]](#references) factored 15 using photonic qubits. By employing a single qubit recycled multiple times and encoding the work register in higher-dimensional states, the researchers reduced the required number of qubits to one-third of that in the standard protocol, utilizing a two-photon compiled algorithm. A significant paper in the demonstration of Shor's algorithm is [[5]](#references), which uses Kitaev's iterative phase estimation [[8]](#references) technique to reduce the qubit requirement of the algorithm. Authors used seven control qubits and four cache qubits, together with the implementation of modular multipliers. This implementation, however, requires mid-circuit measurements with feed-forward operations and qubit recycling with reset operations. This demonstration was done on an ion-trap quantum computer.

More recent work [[6]](#references) focused on factoring 15, 21, and 35 on IBM Quantum&reg; hardware. Similar to previous work, researchers used a compiled version of the algorithm that employed a semi-classical quantum Fourier transform as proposed by Kitaev to minimize the number of physical qubits and gates. A most recent work [[7]](#references) also performed a proof-of-concept demonstration for factoring the integer 21. This demonstration also involved the use of a compiled version of the quantum phase estimation routine, and built upon the previous demonstration by [[4]](#references). Authors went beyond this work by using a configuration of approximate Toffoli gates with residual phase shifts. The algorithm was implemented on IBM quantum processors using only five qubits, and the presence of entanglement between the control and register qubits was verified successfully.

### Scaling of the algorithm

We note that RSA encryption typically involves key sizes on the order of 2048 to 4096 bits. Attempting to factor a 2048-bit number with Shor's algorithm will result in a quantum circuit with millions of qubits, including the error correction overhead and a circuit depth on the order of a billion, which is beyond the limits of current quantum hardware to execute. Therefore, Shor's algorithm will require either optimized circuit construction methods or robust quantum error correction to be practically viable for breaking modern cryptographic systems. We refer you to [[9]](#references) for a more detailed discussion on resource estimation for Shor's algorithm.

## Challenge

Congratulations for finishing the tutorial! Now is a great time to test your understanding. Could you try to construct the circuit for factoring 21? You can select an $a$ of your own choice. You will need to decide on the bit accuracy of the algorithm to choose the number of qubits, and you will need to design the modular exponentiation operators $M_a$. We encourage you to try this out yourself, and then read about the methodologies shown in Fig. 9 of [[6]](#references) and Fig. 2 of [[7]](#references).

In [ ]:
def M_a_mod21():
    """
    M_a (mod 21)
    """

    # Your code here
    pass

훨씬 깔끔합니다. 위수(r)는 N보다 작아야 하므로, 최대 분모를 `15`로 설정합니다: